In [22]:
from NumOpt import ca

x=ca.MX.sym("x")
func=ca.Function("func",[x],[ca.sin(x)],["x"],["y"])

ca.GraphBuilder(func).export_onnx("func.onnx")

In [23]:
g=ca.GraphBuilder("func.onnx").create("func", {"symbolic": True})


In [24]:
g(x=30/180*ca.pi)

{'y': DM(0.5)}

In [27]:
import numpy
from onnx import helper, TensorProto, numpy_helper  # type: ignore
import onnx 
import os 
import tempfile

ELEM = {numpy.float32: TensorProto.FLOAT, numpy.float64: TensorProto.DOUBLE}

def save_model(graph):
# Serialize a GraphProto to a temporary .onnx file and return its path
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 13)])
    model.ir_version = 8
    onnx.checker.check_model(model)
    onnx.save(model, "test.onnx")

def affine_model(dtype, shape, a, b):
# y = a*x + b, elementwise, over a tensor of the given shape and dtype
    et = ELEM[dtype]
    av = numpy_helper.from_array(numpy.full(shape, a, dtype), "a")
    bv = numpy_helper.from_array(numpy.full(shape, b, dtype), "b")
    x = helper.make_tensor_value_info("x", et, list(shape))
    y = helper.make_tensor_value_info("y", et, list(shape))
    g = helper.make_graph(
        [helper.make_node("Mul", ["x", "a"], ["t"]),
            helper.make_node("Add", ["t", "b"], ["y"])],
        "affine", [x], [y], [av, bv])
    return save_model(g)

In [28]:
path = affine_model(numpy.float32, (3,), 2.0, 1.0)

In [27]:
from NumOpt import ca ,np 

def quasi_uniform_knots(ncpts, degree):
    middle = np.linspace(0, 1, ncpts - degree + 1)
    start = np.zeros(degree, dtype="f8")
    end = np.ones(degree, dtype="f8")
    return np.hstack([start, middle, end])


knots=quasi_uniform_knots(ncpts=4,degree=3)
cpts=np.array([
    [0.0,0.0],
    [1.0,2.0],
    [2.0,3.5],
    [1.0,-0.5]
])

u=ca.MX.sym("u")
sp=ca.Function("sp",[u],[ca.bspline(u,cpts.T,[knots],[3,],2)])
sp(ca.DM([[1.0,0.0]])).T



DM(
[[1, -0.5], 
 [0, 0]])

In [ ]:
ca.interpolant()